In [ ]:
import numpy as np
import rasterio as rio
import itertools
from affine import Affine
from rasterio.enums import Resampling
from rasterio.warp import reproject
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

# 3 — Resample IUCN habitat-class maps to 1 km

Merges the former `4-resample-habitat-class-map-1` (level 1) and `-2` (level 2): one parametrized routine. Each level of the Jung et al. (2020) global habitat map is majority (block-mode) downsampled ~100 m → ~1 km, then reprojected to the 1 km Eckert IV grid.

Outputs: `terrestrial-habitat-class-maps/habitat-classifications-lv{1,2}[-1km].tif`. (AOH uses level 2.)

In [ ]:
def block_mode_fast(arr, block_size):
    H, W = arr.shape
    r = block_size
    assert H % r == 0 and W % r == 0, 'Dimensions must be divisible by block size'
    reshaped = arr.reshape(H // r, r, W // r, r)
    blocks = reshaped.transpose(0, 2, 1, 3)
    flat_blocks = blocks.reshape(H // r, W // r, -1)
    max_val = arr.max()
    modes = np.empty((H // r, W // r), dtype=arr.dtype)
    for i in range(H // r):
        for j in range(W // r):
            modes[i, j] = np.bincount(flat_blocks[i, j], minlength=max_val + 1).argmax()
    return modes

In [ ]:
def resample_habitat_map(level):
    """Downsample the Jung et al. (2020) IUCN habitat map (level 1 or 2) to ~1 km by majority
    (block mode), then reproject to the 1 km Eckert IV grid. level in {1, 2}."""
    src = rio.open(DATA / 'a-global-map-of-terrestrial-habitat-types' / f'iucn_habitatclassification_composite_lvl{level}_ver004.tif')
    H, W = src.shape
    k, r = 10000, 10
    q = int(k // r)
    I, J = H // k + 1, W // k + 1
    new_array = np.zeros((I * q, J * q), dtype=np.uint16)
    for i, j in tqdm(itertools.product(range(I), range(J)), desc=f'lvl{level} block-mode'):
        m = k if i < (I - 1) else H - i * k
        n = k if j < (J - 1) else W - j * k
        a_slice = src.read(1, window=((i*k, i*k+m), (j*k, j*k+n)))
        aa = np.vstack([a_slice, np.zeros((k - a_slice.shape[0], a_slice.shape[1]))]).astype(np.uint16)
        bb = np.hstack([aa, np.zeros((aa.shape[0], k - a_slice.shape[1]))]).astype(np.uint16)
        cc = np.where(bb > 0, bb, 0).astype(np.uint16)
        new_array[i*q:(i+1)*q, j*q:(j+1)*q] = block_mode_fast(cc, block_size=10)
    the_transform = Affine(src.transform[0]*10, 0.0, src.transform[2], 0.0, -src.transform[0]*10, src.transform[5])
    coarse_path = RESULTS / 'terrestrial-habitat-class-maps' / f'habitat-classifications-lv{level}.tif'
    with rio.open(coarse_path, 'w', driver='GTiff', height=20038, width=40076, count=1,
                  dtype=new_array.dtype, crs='EPSG:4326', transform=the_transform, compress='lzw') as dst:
        dst.write(new_array[:20038, :40076], 1)
    # reproject to 1 km Eckert IV
    coarse_ = rio.open(coarse_path)
    country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
    out_1km = np.zeros(country_raster_1km_.shape, dtype=np.uint16)
    reproject(source=coarse_.read(1), destination=out_1km,
              src_transform=coarse_.transform, src_crs='EPSG:4326',
              dst_transform=country_raster_1km_.transform, dst_crs=country_raster_1km_.crs,
              resampling=Resampling.mode)
    with rio.open(RESULTS / 'terrestrial-habitat-class-maps' / f'habitat-classifications-lv{level}-1km.tif', 'w',
                  driver='GTiff', height=out_1km.shape[0], width=out_1km.shape[1], count=1,
                  dtype=out_1km.dtype, crs=country_raster_1km_.crs,
                  transform=country_raster_1km_.transform, compress='lzw') as dst:
        dst.write(out_1km, 1)

## Run — both habitat levels

In [ ]:
Path(RESULTS / 'terrestrial-habitat-class-maps').mkdir(parents=True, exist_ok=True)
for level in [1, 2]:
    resample_habitat_map(level)